In [ ]:
import shutil
import zipfile
import rarfile
import py7zr
import subprocess
from pathlib import Path

#SOURCE_ARCHIVE = Path(".zip")
#TARGET = Path("Источники информации_распаковано")

SEVEN_ZIP = r"C:\Program Files\7-Zip\7z.exe"

ARCHIVE_EXTS = {".zip", ".rar", ".7z"}




def prepare():
    if TARGET.exists():
        shutil.rmtree(TARGET)

    TARGET.mkdir()

    with zipfile.ZipFile(SOURCE_ARCHIVE) as z:
        z.extractall(TARGET)



# определение многотомных

def is_multivolume(path: Path) -> bool:
    name = path.name.lower()
    return (
        ".001" in name
        or ".part" in name
        or name.endswith(".rar") and "part" in name
    )



# распаковка через 7z.exe

def extract_with_7z(archive: Path):
    dest = archive.parent / archive.stem
    dest.mkdir(exist_ok=True)

    cmd = [
        SEVEN_ZIP,
        "x",
        str(archive),
        f"-o{dest}",
        "-y",
    ]

    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)



# zip (пофайлово, чтобы не падал на битых)

def extract_zip(archive: Path, dest: Path):
    with zipfile.ZipFile(archive) as z:
        for m in z.infolist():
            try:
                z.extract(m, dest)
            except:
                pass




def extract_rar(archive: Path, dest: Path):
    with rarfile.RarFile(archive) as r:
        for m in r.infolist():
            try:
                r.extract(m, dest)
            except:
                pass





def extract_7z(archive: Path, dest: Path):
    try:
        with py7zr.SevenZipFile(archive, "r") as s:
            s.extractall(dest)
    except:
        pass





def extract_archive(archive: Path):
    dest = archive.parent / archive.stem
    dest.mkdir(exist_ok=True)

    ext = archive.suffix.lower()

    try:
        if is_multivolume(archive):
            extract_with_7z(archive)
        else:
            if ext == ".zip":
                extract_zip(archive, dest)
            elif ext == ".rar":
                extract_rar(archive, dest)
            elif ext == ".7z":
                extract_7z(archive, dest)
            else:
                return

        archive.unlink()

    except Exception as e:
        print("ошибка:", archive, e)
        try:
            archive.unlink()
        except:
            pass





def unpack_all():

    iteration = 1

    while True:

        archives = [
            p for p in TARGET.rglob("*")
            if p.is_file() and (
                p.suffix.lower() in ARCHIVE_EXTS
                or is_multivolume(p)
            )
        ]

        if not archives:
            break

        print(f"\nитерация {iteration}, архивов: {len(archives)}")

        for a in archives:
            print("→", a.relative_to(TARGET))
            extract_archive(a)

        iteration += 1



def clean_dirs():

    while True:

        removed = False

        for d in sorted(TARGET.rglob("*"), reverse=True):
            if d.is_dir():
                try:
                    d.rmdir()
                    removed = True
                except:
                    pass

        if not removed:
            break




if __name__ == "__main__":
    prepare()
    unpack_all()
    clean_dirs()
    print("готово")